In [2]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    filename="logs/infra_10311/mgmt_operations.infra_10311.renames.log",
    encoding="utf-8",
)
logging.getLogger('epicsarchiver').setLevel(logging.DEBUG)
LOG: logging.Logger = logging.getLogger(__name__)

In [3]:
renames = [
("BPROD:Ops:AllowedBModeBDst",	"NSO-BProd::AllowedBModeBDst"),
("BPROD:Ops:BParameters",	"NSO-BProd::BParameters"),
("BPROD:Ops:BSourcePL-RB",	"NSO-BProd::BSourcePL-RB"),
("BPROD:Ops:BLebtPL-RB",	"NSO-BProd::BLebtPL-RB"),
("BPROD:Ops:BeamPL-RB",	"NSO-BProd::BeamPL-RB"),
("BPROD:Ops:BParamsPL",	"NSO-BProd::BParamsPL"),
("BPROD:Ops:BParamsI",	"NSO-BProd::BParamsCurr"),
("BPROD:Ops:BParamsF",	"NSO-BProd::BParamsFreq"),
("BPROD:Ops:BMode",	    "NSO-BProd::BMode"),
("BPROD:Ops:BState",	"NSO-BProd::BState"),
("BPROD:Ops:BDestination",	"NSO-BProd::BDestination"),
]

In [5]:
from epicsarchiver.mgmt.archiver_mgmt_info import ArchivingStatus
from epicsarchiver.mgmt.archiver_mgmt_operations import Storage
from epicsarchiver import ArchiverAppliance

def rename_pv(archiver, old_pv, new_pv) -> dict[str, dict[str, str]]:
    LOG.info(f"Renaming {old_pv} to {new_pv}")
    new_pv_status = archiver.get_archiving_status(new_pv)
    LOG.info(f"New PV {new_pv} status: {new_pv_status}")
    archiver.pause_rename_resume_pv(old_pv, new_pv)
    LOG.info(f"Renamed {old_pv} to {new_pv}")
    new_pv_status = archiver.get_archiving_status(new_pv)
    if new_pv_status == ArchivingStatus.Paused:
        LOG.info(f"Resuming {new_pv}")
        archiver.resume_pv(new_pv)
    else:
        return {new_pv: {"status": new_pv_status, "old_pv": old_pv}}
    return {}

def rename_pvs(archiver, renames: list[tuple[str, str]]):
    for old_pv, new_pv in renames:
        rename_pv(archiver, old_pv, new_pv)

In [6]:
archiver_tn_04_linac = ArchiverAppliance("archiver-linac-04.tn.esss.lu.se")

In [7]:
results = rename_pvs(archiver_tn_04_linac, renames)

In [8]:
print(results)

None
